# 1. Documents and document loaders

In [17]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

print(documents)

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'), Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]


## Loading Documents

In [18]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Noor_Ul_Amin_CV.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

3
Noor Ul Amin
Date of birth: 01/04/1996  | Nationality: Pakistani  | Gender: Male  | Phone: (+92)
3322696097 (Mobile)  | Email address: noorawan444@gmail.com  | Website:
https://my-portfolio-alpha-ashe

{'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2026-02-26T16:09:38+05:00', 'title': 'Europass CV', 'moddate': '2026-02-26T11:09:38+00:00', 'source': 'Noor_Ul_Amin_CV.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## Splitting

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

7


# 2. Embeddings

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 258.67it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.05978229641914368, 0.1133531853556633, -0.04582694172859192, -0.02290293388068676, -0.013882437720894814, -0.03991420194506645, 0.052809011191129684, -0.027866704389452934, 0.005951205268502235, -0.0027509774081408978]


# 3. Vector stores

In [22]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [23]:
ids = vector_store.add_documents(documents=all_splits)

In [24]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='Jul 2021 – Apr 2023
•  Architected scalable backend services for a multi-tenant enterprise platform.
•  Designed secure RBAC authentication and authorization using Auth0.
•  Built workflow-driven APIs for vendor onboarding, resource management, and
approvals.
•  Integrated real-time communication using Twilio Conversations API.
•  Developed responsive frontend using React, Reactstrap, Redux, and Redux-Saga.
•  Added unit and integration testing to improve system reliability.
•  Contributed across the full product lifecycle including requirements, design reviews,
development, and code reviews.
Tech Stack: Node.js, Express.js, MySQL, Sequelize, React, Redux, Auth0, Twilio
Email: noor.amin@productbox.dev  | Website: https://productbox.dev/
01/08/2020 - 30/06/2021  -  PESHAWAR, PAKISTAN
MEVN STACK DEVELOPER NEXT GENERATION CIRCLE (NGEN)
Product: PlantsEra — Full-Stack E-commerce Platform
Aug 2020 – Jun 2021
•  Developed product catalog, cart, order, and user management system

In [25]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score(
    "What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.25960053541223554

page_content='Jul 2021 – Apr 2023
•  Architected scalable backend services for a multi-tenant enterprise platform.
•  Designed secure RBAC authentication and authorization using Auth0.
•  Built workflow-driven APIs for vendor onboarding, resource management, and
approvals.
•  Integrated real-time communication using Twilio Conversations API.
•  Developed responsive frontend using React, Reactstrap, Redux, and Redux-Saga.
•  Added unit and integration testing to improve system reliability.
•  Contributed across the full product lifecycle including requirements, design reviews,
development, and code reviews.
Tech Stack: Node.js, Express.js, MySQL, Sequelize, React, Redux, Auth0, Twilio
Email: noor.amin@productbox.dev  | Website: https://productbox.dev/
01/08/2020 - 30/06/2021  -  PESHAWAR, PAKISTAN
MEVN STACK DEVELOPER NEXT GENERATION CIRCLE (NGEN)
Product: PlantsEra — Full-Stack E-commerce Platform
Aug 2020 – Jun 2021
•  Developed product catalog, cart, order

# 4. Retrievers

In [26]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='f7c3c454-b90e-4288-94ae-f01a9672fa3d', metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2026-02-26T16:09:38+05:00', 'title': 'Europass CV', 'moddate': '2026-02-26T11:09:38+00:00', 'source': 'Noor_Ul_Amin_CV.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'start_index': 0}, page_content='Jul\xa02021\xa0–\xa0Apr\xa02023\n•\xa0\xa0Architected\xa0scalable\xa0backend\xa0services\xa0for\xa0a\xa0multi-tenant\xa0enterprise\xa0platform.\n•\xa0\xa0Designed\xa0secure\xa0RBAC\xa0authentication\xa0and\xa0authorization\xa0using\xa0Auth0.\n•\xa0\xa0Built\xa0workflow-driven\xa0APIs\xa0for\xa0vendor\xa0onboarding,\xa0resource\xa0management,\xa0and\napprovals.\n•\xa0\xa0Integrated\xa0real-time\xa0communication\xa0using\xa0Twilio\xa0Conversations\xa0API.\n•\xa0\xa0Developed\xa0responsive\xa0frontend\xa0using\xa0React,\xa0Reactstrap,\xa0Redux,\xa0and\xa0Redux-Saga.\n•\xa0\xa0Added\xa0unit\